In [ ]:
# ============================================================
# KG1 v51 DIAGNOSTIC — Captura stack trace + treino com loop manual
# REINICIE O RUNTIME ANTES DE RODAR
# ============================================================

!pip install -q peft datasets accelerate trl huggingface_hub safetensors pandas

import subprocess, sys, os, json, random, time, zipfile, shutil, re, math, types, gc
import importlib, importlib.machinery, traceback
from datetime import datetime, timezone
from collections import Counter

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ==================== STUBS ====================
class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None
    def __getattr__(self, name): return _Stub()

for pkg in ['mamba_ssm', 'mamba_ssm.ops', 'mamba_ssm.ops.triton',
            'mamba_ssm.ops.triton.layernorm_gated',
            'mamba_ssm.ops.triton.selective_state_update',
            'mamba_ssm.ops.triton.ssd_combined',
            'mamba_ssm.utils', 'mamba_ssm.utils.generation',
            'causal_conv1d', 'causal_conv1d.causal_conv1d_interface']:
    if pkg not in sys.modules:
        m = types.ModuleType(pkg)
        m.__version__ = '0.0.0'
        m.__spec__ = importlib.machinery.ModuleSpec(pkg, None)
        m.__path__ = []
        m.__file__ = 'stub'
        for attr in ['RMSNormGated', 'rmsnorm_fn', 'selective_state_update',
                     'mamba_chunk_scan_combined', 'mamba_split_conv1d_scan_combined',
                     'InferenceParams', 'GenerationMixin',
                     'causal_conv1d_fn', 'causal_conv1d_update']:
            setattr(m, attr, _Stub)
        sys.modules[pkg] = m

ms = sys.modules['mamba_ssm']
ms.ops = sys.modules['mamba_ssm.ops']
ms.ops.triton = sys.modules['mamba_ssm.ops.triton']
ms.ops.triton.layernorm_gated = sys.modules['mamba_ssm.ops.triton.layernorm_gated']
ms.ops.triton.selective_state_update = sys.modules['mamba_ssm.ops.triton.selective_state_update']
ms.ops.triton.ssd_combined = sys.modules['mamba_ssm.ops.triton.ssd_combined']
ms.utils = sys.modules['mamba_ssm.utils']
ms.utils.generation = sys.modules['mamba_ssm.utils.generation']
print('Stubs OK')

# ==================== PYTORCH + PATCHES ====================
import torch

# Patch F.linear UMA VEZ
if not hasattr(torch.nn.functional, '_real_linear'):
    torch.nn.functional._real_linear = torch.nn.functional.linear
def _safe_linear(input, weight, bias=None):
    if not isinstance(input, torch.Tensor):
        input = torch.zeros(1, weight.shape[-1], dtype=weight.dtype, device=weight.device)
    return torch.nn.functional._real_linear(input, weight, bias)
torch.nn.functional.linear = _safe_linear
torch.nn.Linear.forward = lambda self, input: _safe_linear(input, self.weight, self.bias)
print('F.linear patch OK')

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False

import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

def _get_secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v: return v
    except Exception: pass
    for n in names:
        v = os.environ.get(n)
        if v: return v
    return ''

HF_TOKEN = _get_secret('HF_KEY', 'HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN); os.environ['HF_TOKEN'] = HF_TOKEN; print('HF login OK')

KAGGLE_USERNAME = _get_secret('KAGGLE_USERNAME') or 'felipe1983'
KAGGLE_KEY = _get_secret('KAGGLE_KEY')
if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kpath = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kpath, 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print(f'Kaggle: {KAGGLE_USERNAME}')

api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
DATA_REPO = 'felipesp1983/kg1-nemotron-training'
OUTPUT_REPO = 'felipesp1983/kg1-nemotron-lora-v51-perfect'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = '378df16e4b54'
COMPETITION = 'nvidia-nemotron-model-reasoning-challenge'
N_EXAMPLES = 5000; N_EPOCHS = 2
CONFIG = {
    'lora_rank': 32, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'target_modules': 'all-linear', 'learning_rate': 5e-5,
    'per_device_batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 1024, 'warmup_ratio': 0.05, 'weight_decay': 0.01,
    'lr_scheduler': 'cosine', 'optim': 'adamw_torch',
    'output_dir': '/tmp/kg1_output/v51',
}

try:
    for p in ['/usr/local/cuda-12.8/bin/ptxas', '/usr/local/cuda/bin/ptxas']:
        if os.path.exists(p):
            t = os.path.join(os.path.dirname(shutil.which('python') or '/usr/bin/python'), 'ptxas')
            if not os.path.exists(t): shutil.copy2(p, t)
            break
except Exception: pass

# ==================== LOAD DATA ====================
print('\n=== Loading data ===')
all_examples = []
try:
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/sft_v51_perfect.jsonl', local_dir='/tmp/kg1_data')
    with open('/tmp/kg1_data/data/sft_v51_perfect.jsonl') as f:
        for line in f: all_examples.append(json.loads(line))
    print(f'Loaded: {len(all_examples)} examples')
except Exception as e:
    print(f'Failed ({e}), using train.csv')
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset', filename='data/train.csv', local_dir='/tmp/kg1_data')
    for _, row in pd.read_csv('/tmp/kg1_data/data/train.csv').iterrows():
        all_examples.append({'prompt': row['prompt']+'\nPut your final answer inside \\boxed{}.', 'completion': '\\boxed{'+str(row['answer'])+'}', 'family': 'unknown'})

def classify(t):
    p = t.lower()
    if 'bit manipulation' in p: return 'bit'
    if 'gravitational' in p: return 'grav'
    if 'unit conversion' in p or 'measurement' in p: return 'unit'
    if 'numeral' in p: return 'num'
    if 'encryption' in p: return 'enc'
    if 'transformation' in p: return 'eq'
    return 'other'

random.seed(42)
by_family = {}
for ex in all_examples:
    fam = ex.get('family') or classify(ex.get('prompt', ''))
    by_family.setdefault(fam, []).append(ex)
shares = {'grav':1,'unit':1,'num':1,'enc':1,'cipher':1,'bit':1.5,'eq':2.5,'equation':2.5,'gravity':1,'numeral':1}
total_shares = sum(shares.get(f, 1.0) for f in by_family)
base_n = N_EXAMPLES / total_shares
examples = []
for fam, pool in by_family.items():
    n = int(base_n * shares.get(fam, 1.0))
    if n <= len(pool): examples.extend(random.sample(pool, n))
    else: examples.extend(pool + random.choices(pool, k=n - len(pool)))
random.shuffle(examples)
examples = examples[:N_EXAMPLES]
formatted = [{'messages': [{'role': 'user', 'content': ex.get('prompt', '')}, {'role': 'assistant', 'content': ex.get('completion', '')}]} for ex in examples]
print(f'Dataset: {len(formatted)} examples')

# ==================== LOAD MODEL ====================
print(f'\n=== Loading model (revision {MODEL_REVISION}) ===')
gc.collect(); torch.cuda.empty_cache()
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION,
    device_map={'': 0}, trust_remote_code=True, torch_dtype=torch.bfloat16, low_cpu_mem_usage=True)
for module in model.modules():
    if hasattr(module, 'is_fast_path_available'): module.is_fast_path_available = False
print(f'Model: {model.num_parameters()/1e9:.1f}B, VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# LoRA
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'], target_modules=CONFIG['target_modules'], bias='none', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

# Desabilitar gradient checkpointing explicitamente
model.gradient_checkpointing_disable()
print('Gradient checkpointing: DISABLED')

# ==================== DIAGNOSTICO: STACK TRACE ====================
print('\n=== DIAGNOSTICO: Capturando stack trace ===')

texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in formatted[:10]]
batch = tokenizer(texts[0], return_tensors='pt', truncation=True, max_length=1024, padding=True).to('cuda')

print('Tentando forward+backward com 1 sample...')
try:
    model.train()
    out = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=batch['input_ids'])
    print(f'Forward OK! loss={out.loss.item():.4f}')
    out.loss.backward()
    print('Backward OK!')
    model.zero_grad()
    MANUAL_TRAINING = False
    print('\n>>> SFTTrainer deveria funcionar tambem!')
except Exception as e:
    print(f'\n!!! ERRO no forward/backward: {e}')
    print('\n=== STACK TRACE COMPLETO ===')
    traceback.print_exc()
    print('=== FIM ===')
    MANUAL_TRAINING = True
    print('\n>>> Vou tentar treino com loop manual...')

# ==================== TREINO ====================
if not MANUAL_TRAINING:
    # SFTTrainer funciona
    print('\n=== Training com SFTTrainer ===')
    import inspect
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback
    os.makedirs(CONFIG['output_dir'], exist_ok=True)

    texts_all = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in formatted]
    ds = Dataset.from_dict({'text': texts_all})

    sft_params = inspect.signature(SFTConfig).parameters
    length_key = 'max_seq_length' if 'max_seq_length' in sft_params else 'max_length'
    tok_key = 'processing_class' if 'processing_class' in inspect.signature(SFTTrainer).parameters else 'tokenizer'

    training_args = SFTConfig(**{
        'output_dir': CONFIG['output_dir'], 'dataset_text_field': 'text',
        length_key: CONFIG['max_length'], 'packing': False,
        'num_train_epochs': N_EPOCHS,
        'per_device_train_batch_size': CONFIG['per_device_batch_size'],
        'gradient_accumulation_steps': CONFIG['gradient_accumulation_steps'],
        'learning_rate': CONFIG['learning_rate'], 'warmup_ratio': CONFIG['warmup_ratio'],
        'weight_decay': CONFIG['weight_decay'], 'lr_scheduler_type': CONFIG['lr_scheduler'],
        'optim': CONFIG['optim'], 'bf16': True, 'logging_steps': 5,
        'save_strategy': 'steps', 'save_steps': 100, 'save_total_limit': 15,
        'gradient_checkpointing': False,
        'report_to': 'none', 'dataloader_num_workers': 0, 'max_grad_norm': 1.0,
    })

    class UploadCB(TrainerCallback):
        def __init__(self, repo):
            self.repo = repo; self.hf = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
            try: self.hf.create_repo(repo, private=True, exist_ok=True)
            except: pass
        def on_save(self, args, state, control, **kw):
            import glob as g
            step = state.global_step; loss = 'N/A'
            if state.log_history:
                for e in reversed(state.log_history):
                    if 'loss' in e: loss = e['loss']; break
            ckpts = sorted(g.glob(f'{args.output_dir}/checkpoint-*'))
            if ckpts:
                try:
                    self.hf.upload_folder(folder_path=ckpts[-1], path_in_repo=f'checkpoint-{step}',
                        repo_id=self.repo, commit_message=f'Step {step} Loss {loss}')
                    print(f'\n>>> HF OK: step {step}, loss={loss}')
                except Exception as e: print(f'\n>>> HF FAIL: {e}')

    trainer = SFTTrainer(model=model, train_dataset=ds, **{tok_key: tokenizer},
        args=training_args, callbacks=[UploadCB(OUTPUT_REPO)])
    print(f'Steps: ~{(len(ds)//CONFIG["gradient_accumulation_steps"])*N_EPOCHS}')
    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    print(f'Done: {elapsed/3600:.2f}h')

else:
    # Loop manual de treino (bypass SFTTrainer)
    print('\n=== Training com LOOP MANUAL (bypass SFTTrainer) ===')
    from torch.utils.data import DataLoader
    from torch.optim import AdamW
    from transformers import get_cosine_schedule_with_warmup
    os.makedirs(CONFIG['output_dir'], exist_ok=True)

    # Tokenizar
    all_texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in formatted]
    all_encodings = [tokenizer(t, return_tensors='pt', truncation=True, max_length=CONFIG['max_length'], padding='max_length') for t in all_texts]

    class SimpleDataset(torch.utils.data.Dataset):
        def __init__(self, encodings): self.encodings = encodings
        def __len__(self): return len(self.encodings)
        def __getitem__(self, idx):
            enc = self.encodings[idx]
            return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze(),
                    'labels': enc['input_ids'].squeeze()}

    train_ds = SimpleDataset(all_encodings)
    train_loader = DataLoader(train_ds, batch_size=CONFIG['per_device_batch_size'], shuffle=True)

    optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
    total_steps = len(train_loader) * N_EPOCHS // CONFIG['gradient_accumulation_steps']
    warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    print(f'Total steps: {total_steps}, warmup: {warmup_steps}')
    model.train()
    global_step = 0
    accum_loss = 0.0
    start = time.time()
    hf_api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
    try: hf_api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    except: pass

    for epoch in range(N_EPOCHS):
        for step_in_batch, batch in enumerate(train_loader):
            batch = {k: v.to('cuda') for k, v in batch.items()}
            try:
                outputs = model(**batch)
                loss = outputs.loss / CONFIG['gradient_accumulation_steps']
                loss.backward()
                accum_loss += loss.item()
            except Exception as e:
                print(f'\n!!! Step {global_step} error: {e}')
                traceback.print_exc()
                continue

            if (step_in_batch + 1) % CONFIG['gradient_accumulation_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                if global_step % 5 == 0:
                    avg_loss = accum_loss / 5 if global_step >= 5 else accum_loss
                    elapsed_h = (time.time() - start) / 3600
                    print(f'Step {global_step}/{total_steps} | Loss: {avg_loss:.4f} | Epoch: {epoch+1}/{N_EPOCHS} | Time: {elapsed_h:.2f}h')
                    accum_loss = 0.0

                if global_step % 100 == 0:
                    ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-{global_step}'
                    model.save_pretrained(ckpt_dir)
                    tokenizer.save_pretrained(ckpt_dir)
                    print(f'  Saved checkpoint-{global_step}')
                    try:
                        hf_api.upload_folder(folder_path=ckpt_dir, path_in_repo=f'checkpoint-{global_step}',
                            repo_id=OUTPUT_REPO, commit_message=f'Step {global_step}')
                        print(f'  HF upload OK')
                    except: print(f'  HF upload failed')

    elapsed = time.time() - start
    print(f'\nTraining done: {elapsed/3600:.2f}h, {global_step} steps')

# ==================== SAVE ====================
print('\n=== Saving ===')
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
        commit_message=f'v51 final')
    print('Uploaded to HF')
except Exception as e: print(f'Upload failed: {e}')

# ==================== SMART STRIP + SUBMIT ====================
print('\n=== Smart Strip ===')
import glob
from safetensors.torch import load_file, save_file
ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-400'
if not os.path.exists(ckpt_dir):
    ckpts = sorted(glob.glob(f'{CONFIG["output_dir"]}/checkpoint-*'), key=lambda x: int(x.split('-')[-1]))
    ckpt_dir = ckpts[-1] if ckpts else CONFIG['output_dir']
    print(f'Using: {ckpt_dir}')
tensors = load_file(os.path.join(ckpt_dir, 'adapter_model.safetensors'))
routed_re = re.compile(r'\.experts\.\d+\.')
keep = {k: v for k, v in tensors.items() if not routed_re.search(k)}
print(f'Kept: {len(keep)} | Removed: {len(tensors) - len(keep)}')
out_dir = '/tmp/kg1_submit/stripped'
os.makedirs(out_dir, exist_ok=True)
save_file(keep, os.path.join(out_dir, 'adapter_model.safetensors'))
with open(os.path.join(ckpt_dir, 'adapter_config.json')) as f: cfg = json.load(f)
mods = set()
for k in keep:
    for m in ['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','gate']:
        if m in k: mods.add(m)
cfg['target_modules'] = sorted(mods)
with open(os.path.join(out_dir, 'adapter_config.json'), 'w') as f: json.dump(cfg, f, indent=2)
zip_path = '/tmp/kg1_submit/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(out_dir, fn), fn)
print(f'ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')
step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-step{step_str}-smart-strip'
print(f'Submitting: {desc}')
os.system(f'kaggle competitions submit -c {COMPETITION} -f {zip_path} -m "{desc}"')
print(f'\nDONE: {desc}')
